<div class="align-center">
<a href="https://oumi.ai/"><img src="https://oumi.ai/docs/en/latest/_static/logo/header_logo.png" height="200"></a>

[![Documentation](https://img.shields.io/badge/Documentation-latest-blue.svg)](https://oumi.ai/docs/en/latest/index.html)
[![Discord](https://img.shields.io/discord/1286348126797430814?label=Discord)](https://discord.gg/oumi)
[![GitHub Repo stars](https://img.shields.io/github/stars/oumi-ai/oumi)](https://github.com/oumi-ai/oumi)
</div>

👋 Welcome to Open Universal Machine Intelligence (Oumi)!

🚀 Oumi is a fully open-source platform that streamlines the entire lifecycle of foundation models — from [data preparation](https://oumi.ai/docs/en/latest/resources/datasets/datasets.html) and [training](https://oumi.ai/docs/en/latest/user_guides/train/train.html) to [evaluation](https://oumi.ai/docs/en/latest/user_guides/evaluate/evaluate.html) and [deployment](https://oumi.ai/docs/en/latest/user_guides/launch/launch.html).

🤝 Make sure to join our [Discord community](https://discord.gg/oumi) to get help, share your experiences, and contribute to the project!

⭐ If you like Oumi and you would like to support it, please give it a star on [GitHub](https://github.com/oumi-ai/oumi).

# AIDE Custom Evaluation Design

Building custom evaluation functions is one of the most time-consuming parts of ML development. You need to handle prompt construction, response parsing, edge cases, and metric computation. In the [Hallucination Classifier notebook](https://github.com/oumi-ai/oumi/blob/main/notebooks/Oumi%20-%20Build%20your%20own%20Custom%20Evaluation%20(Hallucination%20Classifier).ipynb), this was done by hand.

In this notebook, we use [AIDE (AI-Driven Exploration)](https://github.com/WecoAI/aideml) to **autonomously generate custom evaluation functions**. AIDE writes evaluation code, registers it with Oumi's evaluator, runs it, and iterates until the evaluation logic is robust.

This notebook focuses on the **EVAL_FUNCTION** surface. AIDE supports multiple surfaces:

| Surface | Notebook | Description |
|---------|----------|-------------|
| CONFIG_SEARCH | [AIDE Agentic Optimization](Oumi%20-%20AIDE%20Agentic%20Optimization.ipynb) | Optimize training hyperparameters |
| REWARD_FUNCTION | [AIDE Reward Function Design](Oumi%20-%20AIDE%20Reward%20Function%20Design.ipynb) | Design reward functions for GRPO/RLHF |
| **EVAL_FUNCTION** | **This notebook** | Generate evaluation functions |
| FULL_PIPELINE | Coming soon | Generate complete training scripts |

## Prerequisites

### Machine Requirements

Evaluation requires loading a model for inference. For SmolLM 135M, a single GPU with 8GB+ VRAM is sufficient.

### LLM Access

AIDE needs an LLM to generate and evaluate code. Serve a model locally (free) or use a cloud API.

```bash
# Local model (recommended):
vllm serve Qwen/Qwen3-Coder-Next --port 8000
```

In [ ]:
%pip install uv
!cd .. && uv pip install -e ".[aide]"

# For production: !uv pip install "oumi[aide]"

In [ ]:
import os
from pathlib import Path

# === LLM Access (uncomment one) ===
# os.environ["OPENAI_BASE_URL"] = "http://localhost:8000/v1"  # Local model
# os.environ["OPENAI_API_KEY"] = "dummy"
# os.environ["OPENAI_API_KEY"] = ""      # OpenAI
# os.environ["ANTHROPIC_API_KEY"] = ""   # Anthropic
# os.environ["GEMINI_API_KEY"] = ""      # Gemini

tutorial_dir = "aide_eval_tutorial"
Path(tutorial_dir).mkdir(parents=True, exist_ok=True)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
# Auto-detect LLM provider.
has_local = bool(os.environ.get("OPENAI_BASE_URL"))
has_openai = bool(os.environ.get("OPENAI_API_KEY")) and not has_local
has_anthropic = bool(os.environ.get("ANTHROPIC_API_KEY"))
has_gemini = bool(os.environ.get("GEMINI_API_KEY"))

if has_local:
    CODE_MODEL = FEEDBACK_MODEL = "Qwen/Qwen3-Coder-Next"
    provider = "Local server"
elif has_openai:
    CODE_MODEL, FEEDBACK_MODEL = "o4-mini", "gpt-5-mini"
    provider = "OpenAI"
elif has_anthropic:
    CODE_MODEL = FEEDBACK_MODEL = "claude-sonnet-4-20250514"
    provider = "Anthropic"
elif has_gemini:
    CODE_MODEL = FEEDBACK_MODEL = "gemini-2.5-flash"
    provider = "Gemini"
else:
    CODE_MODEL, FEEDBACK_MODEL = "o4-mini", "gpt-5-mini"
    provider = None

print(f"Provider: {provider or 'None (will show example output)'}")

## How Custom Evaluation Works in Oumi

Oumi's evaluation system uses a registry pattern. You write a function, decorate it with `@register_evaluation_function`, and Oumi's evaluator can call it. Here's the existing letter counting evaluation:

In [ ]:
import inspect

from oumi.evaluation.registry.count_letters_task import count_letters  # type: ignore

# Show the function signature and first part
source = inspect.getsource(count_letters)
print(source[:800])
print("...")

## The Dataset

For this demonstration, we'll use the ANLI (Adversarial NLI) dataset — the same dataset used in the [Hallucination Classifier notebook](https://github.com/oumi-ai/oumi/blob/main/notebooks/Oumi%20-%20Build%20your%20own%20Custom%20Evaluation%20(Hallucination%20Classifier).ipynb). Each example has a premise, a hypothesis, and a label (supported/unsupported).

In [ ]:
import datasets  # type: ignore

anli = datasets.load_dataset("facebook/anli", split="test_r3")
print(f"Dataset size: {len(anli)} examples\n")

# Show a few examples
for i in range(3):
    ex = anli[i]
    label_map = {0: "supported", 1: "unsupported", 2: "unsupported"}
    print(f"Example {i + 1}:")
    print(f"  Premise:    {ex['premise'][:100]}...")
    print(f"  Hypothesis: {ex['hypothesis'][:100]}...")
    print(f"  Label:      {label_map[ex['label']]}")
    print()

### What Makes a Good Evaluation Function?

A custom evaluation function in Oumi must:

```python
@register_evaluation_function('my_eval')
def my_eval(task_params, config, inference_engine):
    # 1. Load or create evaluation dataset
    # 2. Run inference on the dataset
    # 3. Parse model responses
    # 4. Compute metrics
    return {'accuracy': 0.85, 'f1': 0.82}
```

The hard parts are **response parsing** (extracting structured answers from free-text) and **metric design** (choosing what to measure). These are exactly what AIDE can help automate.

### What an Evaluation Function Needs to Do

Building a custom eval is a multi-step process:
1. **Load or create a dataset** with inputs and ground truth labels
2. **Construct prompts** that the model can understand
3. **Run inference** using Oumi's inference engine
4. **Parse model responses** to extract predictions (the hardest part!)
5. **Compute metrics** like accuracy, F1, balanced accuracy

AIDE automates all of this — it generates the entire evaluation function, including the tricky response parsing logic.

> **Note:** This notebook uses the NLI task (pairs with the [Hallucination Classifier notebook](Oumi%20-%20Build%20your%20own%20Custom%20Evaluation%20(Hallucination%20Classifier).ipynb)). You can also try math reasoning by changing the goal to reference GSM8K — Oumi already has `gsm8k_reward.py` that AIDE can build on.

## Let AIDE Generate an Evaluation Function

We configure AIDE with `surface=EVAL_FUNCTION` and describe what the evaluation should test. AIDE's `test_eval()` helper handles registering the function and running it through Oumi's evaluator.

In [ ]:
from oumi.core.configs import (  # type: ignore
    AideConfig,
    AideParams,
    AideSearchParams,
    ModelParams,
)
from oumi.core.configs.params.aide_params import (  # type: ignore
    AideExecParams,
    AideLLMParams,
    AideOptimizationSurface,
)

config = AideConfig(
    model=ModelParams(
        model_name="HuggingFaceTB/SmolLM2-135M-Instruct",
        torch_dtype_str="bfloat16",
        trust_remote_code=True,
    ),
    goal=(
        "Design a custom evaluation function that tests a model's ability "
        "to classify text as supported or unsupported given a premise and "
        "hypothesis (NLI task). Use the facebook/anli dataset. The function "
        "should construct prompts, run inference, extract predictions from "
        "the model's response, and compute balanced accuracy."
    ),
    aide=AideParams(
        steps=5,
        surface=AideOptimizationSurface.EVAL_FUNCTION,
        target_metric="accuracy",
        target_direction="maximize",
        output_dir=f"{tutorial_dir}/output",
        workspace_dir=f"{tutorial_dir}/workspace",
        code_llm=AideLLMParams(model=CODE_MODEL, temperature=0.5),
        feedback_llm=AideLLMParams(model=FEEDBACK_MODEL, temperature=0.5),
        search=AideSearchParams(num_drafts=2),
        execution=AideExecParams(timeout=300),
    ),
)

print(f"Surface:    {config.aide.surface.value}")
print(f"Metric:     {config.aide.target_metric} ({config.aide.target_direction})")
print(f"Code LLM:   {config.aide.code_llm.model}")

In [ ]:
from oumi.aide import aide as run_aide  # type: ignore

if provider:
    result = run_aide(config, verbose=True)

    print(f"\nBest metric: {result.best_metric}")
    print(f"Good: {result.good_solutions}, Buggy: {result.buggy_solutions}")
    print(f"Best solution: {result.best_solution_path}")
else:
    print("[Skipped \u2014 no LLM provider configured]")
    print("\nExample: AIDE generates an eval function that:")
    print("  - Loads ANLI test split")
    print("  - Constructs NLI prompts")
    print("  - Parses <|supported|> / <|unsupported|> tokens")
    print("  - Computes balanced accuracy: 0.52")

## Understanding the Results

AIDE's output includes the generated evaluation function code. Key things to check:
- Does the prompt template make sense for the task?
- Is the response parsing robust (handles edge cases)?
- Are the metrics appropriate (balanced accuracy for imbalanced datasets)?

In [ ]:
# Show the best evaluation function code (if available)
if provider and result.best_code:
    print("--- AIDE's best evaluation function ---")
    print(result.best_code[:2000])
    print("---")

## Summary

In this notebook, you learned how to:

1. ✅ Use AIDE's EVAL_FUNCTION surface to auto-generate evaluation logic
2. ✅ Understand the `@register_evaluation_function` pattern
3. ✅ Let AIDE handle prompt design, response parsing, and metric computation
4. ✅ Analyze the generated evaluation code

## References & Credits

- **AIDE** by [Weco AI](https://www.weco.ai/) — [GitHub](https://github.com/WecoAI/aideml) | [Paper](https://arxiv.org/abs/2502.13138)
- **Oumi** — [Website](https://oumi.ai/) | [GitHub](https://github.com/oumi-ai/oumi) | [Docs](https://oumi.ai/docs/)

# 🧭 What's Next?

Congrats on finishing this notebook! Feel free to check out our other [notebooks](https://github.com/oumi-ai/oumi/tree/main/notebooks) in the [Oumi GitHub](https://github.com/oumi-ai/oumi), and give us a star! You can also join the Oumi community over on [Discord](https://discord.gg/oumi).

- **[AIDE Agentic Optimization](Oumi%20-%20AIDE%20Agentic%20Optimization.ipynb)** — CONFIG_SEARCH surface: optimize hyperparameters
- **[AIDE Reward Function Design](Oumi%20-%20AIDE%20Reward%20Function%20Design.ipynb)** — REWARD_FUNCTION surface: auto-design reward functions
- **[Hallucination Classifier](Oumi%20-%20Build%20your%20own%20Custom%20Evaluation%20(Hallucination%20Classifier).ipynb)** — The manual approach this builds on
- **[Finetuning Tutorial](Oumi%20-%20Finetuning%20Tutorial.ipynb)** — LoRA-tune models with Oumi

Happy optimizing!